In [1]:
import os
import gc
import time
import tempfile
import pandas as pd
from verbatim_rag.index import VerbatimIndex
from verbatim_rag.vector_stores import LocalMilvusStore
from verbatim_rag.embedding_providers import SentenceTransformersProvider
from verbatim_rag.schema import DocumentSchema
from verbatim_rag.rerankers import SentenceTransformersReranker

In [2]:
import sys
sys.path.append("../../")

In [3]:
# Lade die Passages (TSV Datei)
passages_df = pd.read_csv('../../data/passages.tsv', sep='\t')

# Schaue dir die ersten Zeilen an
print("Shape:", passages_df.shape)
print("\nColumns:", passages_df.columns.tolist())
print("\nFirst few rows:")
passages_df.head()

Shape: (178890, 3)

Columns: ['id', 'text', 'title']

First few rows:


,id,text,title
0,827849752_115-357,Egocentrism is the inability to differentiate ...,Egocentrism
1,827849752_358-743,Although egocentrism and narcissism appear sim...,Egocentrism
2,827849752_744-1186,Although egocentric behaviors are less promine...,Egocentrism
3,827849752_1187-1426,"Therefore , egocentrism is found across the li...",Egocentrism
4,827849752_1604-3189,The main concept infants and young children le...,Egocentrism


In [4]:
dense_provider = SentenceTransformersProvider(
    model_name="intfloat/e5-base-v2",
    device="cpu"  # oder "cuda" wenn GPU
)

In [5]:
print("\n🏗️ Building Milvus index for 178k passages (E5-base-v2)...")

db_path = "../../data/clapnq_milvus_e5_v4.db"

vector_store = LocalMilvusStore(
    db_path=db_path,
    collection_name="clapnq_passages_e5",
    enable_dense=True,
    enable_sparse=False,
    dense_dim=768,  # E5-base-v2 hat 768 statt 384
    index_type="FLAT",
)

index = VerbatimIndex(
    vector_store=vector_store,
    dense_provider=dense_provider,
    chunker_provider=None,  # Wir chunken manuell unten
    enhance_embeddings=False,
)


🏗️ Building Milvus index for 178k passages (E5-base-v2)...


/home/pkunerth/projects/verbatim-rag/.venv/lib/python3.12/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [6]:
print("📝 Converting passages to DocumentSchema...")
documents = [
    DocumentSchema(
        content="passage: " + row['title'] + " " + row['text'],
        title=row['title'],
        doc_type="passage",
        metadata={"passage_id": row['id']}
    )
    for _, row in passages_df.iterrows()
]

BATCH_SIZE = 1000
total_batches = (len(documents) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"\n🚀 Indexing {len(documents)} passages in {total_batches} batches...")

for i in range(0, len(documents), BATCH_SIZE):
    batch = documents[i:i+BATCH_SIZE]
    index.add_documents(batch)
    if (i // BATCH_SIZE + 1) % 10 == 0:
        print(f"  ✅ {i+BATCH_SIZE}/{len(documents)}")
    if (i // BATCH_SIZE + 1) % 50 == 0:
        gc.collect()

print(f"\n✅ Done! Indexed {len(documents)} passages.")

📝 Converting passages to DocumentSchema...

🚀 Indexing 178890 passages in 179 batches...


Adding documents: 100%|██████████| 1000/1000 [04:01<00:00,  4.15it/s]


  ✅ 10000/178890


Adding documents: 100%|██████████| 1000/1000 [03:54<00:00,  4.26it/s]


  ✅ 20000/178890


...

  ✅ 160000/178890


Adding documents: 100%|██████████| 1000/1000 [05:38<00:00,  2.95it/s]


  ✅ 170000/178890


Adding documents: 100%|██████████| 890/890 [04:11<00:00,  3.54it/s]


✅ Done! Indexed 178890 passages.


In [7]:
question = "what is the story of call of duty zombie"
results = index.query(text="query: " + question, k=10)

---
## Optional: DB mit Dense + Sparse (Hybrid)

Note: Dazu muss irgendwas noch konfiguriert werden mit Huggingface

Diese Zellen bauen eine **zweite** Datenbank für Sparse- und Hybrid-Retrieval. Die bestehende DB (`clapnq_milvus.db`) bleibt unverändert.

- **Voraussetzung:** Zellen 0–4 zuerst ausführen (Imports, `passages_df`, `dense_provider`, `documents`).
- **Ergebnis:** Neue Datei `data/clapnq_milvus_hybrid.db` mit Collection `clapnq_passages_hybrid` (Dense + Sparse). Indexierung dauert länger wegen SPLADE.
- **Test-Modus:** Es werden nur die ersten **1000** Passages indexiert (`N_HYBRID_TEST = 1000` in Zelle 7). Für den Voll-Lauf später `N_HYBRID_TEST = len(documents)` setzen.

In [8]:
# Sparse-Provider (SPLADE) für Hybrid-DB
from verbatim_rag.embedding_providers import SpladeProvider

sparse_provider = SpladeProvider(
    model_name="naver/splade-v3",
    device="cpu"  # oder "cuda" wenn GPU
)

# Zweite DB: Dense + Sparse (andere Datei & Collection!)
db_path_hybrid = "data/clapnq_milvus_hybrid.db"
vector_store_hybrid = LocalMilvusStore(
    db_path=db_path_hybrid,
    collection_name="clapnq_passages_hybrid",
    enable_dense=True,
    enable_sparse=True,
    dense_dim=dense_provider.get_dimension(),
)

index_hybrid = VerbatimIndex(
    vector_store=vector_store_hybrid,
    dense_provider=dense_provider,
    sparse_provider=sparse_provider,
    chunker_provider=None,
)

# Nur 1000 Passages zum Testen (später z.B. documents für Voll-Lauf)
N_HYBRID_TEST = 1000
documents_hybrid = documents[:N_HYBRID_TEST]
print(f"Store und Index (Dense+Sparse) erstellt. Nutze {len(documents_hybrid)} Passages zum Testen (von {len(documents)}).")

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/naver/splade-v3.
401 Client Error. (Request ID: Root=1-69ddef34-0b7b55c859396265507abbe5;981d3b08-a3dc-4c7b-92ab-33cf303342ee)

Cannot access gated repo for url https://huggingface.co/naver/splade-v3/resolve/main/config.json.
Access to model naver/splade-v3 is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
# Indexierung in die Hybrid-DB (nur documents_hybrid = erste N_HYBRID_TEST Passages)
BATCH_SIZE = 1000
total_batches = (len(documents_hybrid) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Indexing {len(documents_hybrid)} passages into hybrid DB ({total_batches} batches)...")

for i in range(0, len(documents_hybrid), BATCH_SIZE):
    batch = documents_hybrid[i:i + BATCH_SIZE]
    index_hybrid.add_documents(batch)
    if (i // BATCH_SIZE + 1) % 10 == 0:
        print(f"  ✅ Indexed {i + BATCH_SIZE}/{len(documents_hybrid)} passages...")
    if (i // BATCH_SIZE + 1) % 50 == 0:
        gc.collect()

print("\n✅ Hybrid indexing complete!")

In [ ]:
# Kurztest: Dense vs Sparse vs Hybrid auf der neuen DB
question = "what is the story of call of duty zombie"
for st in ["dense", "sparse", "hybrid"]:
    results = index_hybrid.query(text=question, k=3, search_type=st)
    print(f"\n--- {st.upper()} ---")
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r.metadata.get('title', '')} (score: {r.score:.4f})")